# Stochastic Calculus, Step by Step · Lesson 1: The calculus you need

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Derivatives and the chain rule · 3. Taylor's theorem · 4. Why the second-order term usually goes · 5. Integration and substitution · 6. Exponentials and logarithms · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 1,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. Derivatives and the chain rule

A zero-coupon bond paying 1 in $T$ years, discounted continuously at rate $r$, is worth $P(r) = e^{-rT}$. By the chain rule, $P'(r) = -T e^{-rT}$: how much the price moves per unit change in the rate. Check it against finite differences.

In [ ]:
P = lambda rate: math.exp(-rate * T)
exact = -T * math.exp(-r * T)
h = 1e-4
forward = (P(r + h) - P(r)) / h
central = (P(r + h) - P(r - h)) / (2 * h)
out["bond"] = {"P": P(r), "dP": exact, "d2P": T**2 * math.exp(-r * T), "forward": forward, "central": central,
               "forward_err": forward - exact, "central_err": central - exact, "h": h,
               "dP_bp": exact * 1e-4}
# e is the number whose exponential has slope 1 at 0: (e^h - 1)/h -> 1, and (1 + 1/n)^n -> e
out["e_def"] = {"slope": [{"h": hh, "value": (math.exp(hh) - 1) / hh} for hh in (0.1, 0.01, 0.001, 0.0001)],
                "e": math.e, "comp": [{"n": n, "value": (1 + 1 / n) ** n} for n in (1, 10, 100, 10_000)]}
# the picture: e^x near 0, secant lines through (0, 1) shrinking to the tangent y = 1 + x
xs = np.linspace(-1.5, 1.5, 151)
out["e_def"]["fig"] = {"x": xs.round(3).tolist(), "exp": np.exp(xs).round(5).tolist(),
                       **{f"sec_{k}": (1 + (math.exp(hh) - 1) / hh * xs).round(5).tolist()
                          for k, hh in (("1", 1.0), ("05", 0.5))},
                       "tangent": (1 + xs).round(5).tolist(),
                       "ticks": [{"x": v, "label": f"{v:g}"} for v in (-1, -0.5, 0, 0.5, 1, 1.5)]}
out["bond"]

## 3. Taylor's theorem

$\ln(1 + h)$ near $h = 0$: $h - \tfrac12 h^2 + \tfrac13 h^3 - \dots$. A 5% move, $h = 0.05$ (the share going from 100 to 105):

In [ ]:
h = K / S0 - 1
exact = math.log(1 + h)
orders = {"1": h, "2": h - h**2 / 2, "3": h - h**2 / 2 + h**3 / 3}
out["taylor"] = {"h": h, "exact": exact, **{f"o{k}": v for k, v in orders.items()},
                 **{f"e{k}": v - exact for k, v in orders.items()}}
rows = []
for hh in (0.2, 0.1, 0.05, 0.025):
    ex = math.log(1 + hh)
    rows.append({"h": hh, "err1": abs(hh - ex), "err2": abs(hh - hh**2 / 2 - ex)})
for a, b in zip(rows, rows[1:]):
    b["ratio1"], b["ratio2"] = a["err1"] / b["err1"], a["err2"] / b["err2"]
out["taylor"]["rows"] = rows
# the picture: ln x around 1 and its Taylor polynomials
x = np.linspace(0.45, 1.8, 271)
u = x - 1
out["taylor"]["fig"] = {"x": x.round(4).tolist(), "ln": np.log(x).round(5).tolist(), "t1": u.round(5).tolist(),
                        "t2": (u - u**2 / 2).round(5).tolist(), "t3": (u - u**2 / 2 + u**3 / 3).round(5).tolist(),
                        "ticks": [{"x": v, "label": f"{v:g}"} for v in (0.5, 0.75, 1, 1.25, 1.5, 1.75)]}
pd.DataFrame(rows)

## 4. Why the second-order term usually goes

For a small deterministic step $h$, the second-order term $\tfrac12 f''h^2$ is tiny next to $f'h$. Halve the step and it shrinks four times; over $1/h$ steps the squares add up to about $h$, which vanishes. In stochastic calculus the step is random, of size $\sigma\sqrt{\Delta t}$, and its square is of order $\Delta t$: the same size as the drift. Over one trading day:

In [ ]:
dt = 1 / 252
step = sigma * math.sqrt(dt)
out["day"] = {"dt": dt, "step": step, "half_sq": 0.5 * step**2, "drift": r * dt,
              "sum_sq_year": 252 * step**2, "sum_sq_det": 252 * (r * dt) ** 2}
# the same on the 100 share, in dollars and cents
out["day"].update({"step_dollars": S0 * step, "drift_cents": 100 * S0 * r * dt,
                   "half_sq_cents": 100 * S0 * 0.5 * step**2, "ratio": step**2 / (r * dt) ** 2})
out["day"]

## 5. Integration and substitution

Growing at rate $r$, one unit a year paid continuously for a year accumulates to $\int_0^1 e^{rt}\,dt = (e^{r} - 1)/r$. The substitution $u = -x^2/2$ turns $\int_0^\infty x e^{-x^2/2}\,dx$ into $\int_{-\infty}^0 e^u\,du = 1$; Lesson 4 uses it for $E|Z|$. Both checked with the trapezoid rule.

In [ ]:
t = np.linspace(0, 1, 10_001)
g = np.exp(r * t)
trap = float(((g[1:] + g[:-1]) / 2 * np.diff(t)).sum())
x = np.linspace(0, 12, 120_001)
gx = x * np.exp(-x**2 / 2)
trap2 = float(((gx[1:] + gx[:-1]) / 2 * np.diff(x)).sum())
x1 = np.linspace(0, 1, 10_001)
gp = x1 * np.exp(x1)
trap3 = float(((gp[1:] + gp[:-1]) / 2 * np.diff(x1)).sum())
out["integral"] = {"exact": (math.exp(r) - 1) / r, "trap": trap, "sub_exact": 1.0, "sub_trap": trap2,
                   "parts_exact": 1.0, "parts_trap": trap3}
out["integral"]

## 6. Exponentials and logarithms

Continuous compounding is the limit of compounding $n$ times a year; log returns add over time, simple returns don't.

In [ ]:
comp = [{"n": n, "value": S0 * (1 + r / n) ** n} for n in (1, 2, 4, 12, 52, 365)]
growth = S0 * math.exp(r * T)
path = [S0, 110.0, 99.0]
simple = [path[1] / path[0] - 1, path[2] / path[1] - 1]
logs = [math.log(path[1] / path[0]), math.log(path[2] / path[1])]
out["exp"] = {"growth": growth, "simple_growth": S0 * (1 + r * T), "discount_K": K * math.exp(-r * T),
              "comp": comp, "limit_gap": growth - comp[-1]["value"],
              "path": path, "simple": simple, "simple_sum": sum(simple), "logs": logs, "log_sum": sum(logs),
              "total_log": math.log(path[2] / path[0]), "total_simple": path[2] / path[0] - 1}
assert abs(out["exp"]["log_sum"] - out["exp"]["total_log"]) < 1e-15
pd.DataFrame(comp)

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 1: Solutions*.

**Exercise 1.** Differentiate $f(x) = e^{-x^2/2}$ and $g(S) = \ln(S e^{rT})$. What is $g'(100)$?

**Exercise 2.** Write the second-order Taylor polynomial of $\sqrt{x}$ around $x = 1$ and use it to approximate $\sqrt{1.21}$. How big is the error?

**Exercise 3.** Compute $\int_0^T e^{-rt}\,dt$, the value today of 1 a year paid continuously for $T$ years. Evaluate it for $r = 3\%$, $T = 1$.

**Exercise 4.** Show that $(1 + r/n)^n \to e^r$ as $n \to \infty$, using $\ln(1 + x) = x - \tfrac12x^2 + \dots$.

**Exercise 5.** Use integration by parts to compute $\int_0^T t\,e^{-rt}\,dt$. Evaluate it for $r = 3\%$, $T = 1$.

**Exercise 6.** For the bond price $P(r) = e^{-rT}$ with $T = 1$, find $P''(r)$. Estimate the price change when the rate rises from 3% to 4% with the first-order and the second-order Taylor approximations, and compare both with the exact change.

**Exercise 7.** (Code) For $f(x) = e^x$ at $x = 0$, compute the forward-difference and central-difference errors for $h = 10^{-1}, 10^{-2}, \dots, 10^{-10}$. Why does each error fall and then rise again?

**Exercise 8.** (Code) A share goes 100, 110, 99, 108.9. Compute the simple and log returns of each step, and show which kind adds up to the whole period's return.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")